# 04. Model XGBoost
XGBoost models implementation.

In [ ]:
import pandas as pd
import numpy as np
import xgboost as xgb
from sklearn.preprocessing import RobustScaler
import notebook_const

from src import utils
from src import const

In [ ]:
# Create sequences or reuse cached tensors
requested_past_trips = 5
saved_tensors = utils.load_saved_tensors()

if saved_tensors:
    print(f"Using cached tensors from {saved_tensors['meta_path']}")
    metadata = saved_tensors['metadata']
    n_past_trips = int(metadata.get('n_past_trips', requested_past_trips))
    lag_columns = metadata.get('lag_columns', [])
    data = {
        'X_delays_train': saved_tensors['train']['X_delays'],
        'X_features_train': saved_tensors['train']['X_features'],
        'X_agg_train': saved_tensors['train']['X_agg'],
        'y_train': saved_tensors['train']['y'],
        'X_delays_test': saved_tensors['test']['X_delays'],
        'X_features_test': saved_tensors['test']['X_features'],
        'X_agg_test': saved_tensors['test']['X_agg'],
        'y_test': saved_tensors['test']['y'],
        'n_stops': saved_tensors['n_stops'] or saved_tensors['train']['X_delays'].shape[-1],
    }
else:
    df_train, df_valid, df_test, df_process, split_info = utils.load_split_data_with_combined()

    lag_columns = sorted([col for col in df_process.columns if col.startswith('lag_arrival_delay_')])
    if not lag_columns:
        raise RuntimeError("Lag features not found. Please re-run 02_process_data.ipynb to generate lag_arrival_delay_* columns.")
    n_past_trips = min(requested_past_trips, len(lag_columns))
    print(f"Cached tensors not found. Building sequences with {n_past_trips} past trips.")
    data = utils.prepare_model_data(df_train, df_test, df_process, n_past_trips=n_past_trips)

# Extract variables
X_delays_train, X_features_train, X_agg_train, y_train = \
    data['X_delays_train'], data['X_features_train'], data['X_agg_train'], data['y_train']
X_delays_test, X_features_test, X_agg_test, y_test = \
    data['X_delays_test'], data['X_features_test'], data['X_agg_test'], data['y_test']
n_stops = data['n_stops']

In [ ]:
# Prepare Data for XGBoost
X_train_flat = np.concatenate([
    X_delays_train.reshape(len(X_delays_train), -1),
    X_features_train,
    X_agg_train
], axis=1)

X_test_flat = np.concatenate([
    X_delays_test.reshape(len(X_delays_test), -1),
    X_features_test,
    X_agg_test
], axis=1)

scaler = RobustScaler()
X_train_scaled = scaler.fit_transform(X_train_flat)
X_test_scaled = scaler.transform(X_test_flat)

In [ ]:
# Train XGBoost (Per Stop)
evaluation_results = []

print("Training XGBoost...")
y_pred_xgb_all = []

xgb_params = {
    "n_estimators": 100,
    "max_depth": 5,
    "learning_rate": 0.1,
    "n_jobs": -1,
    "random_state": 42
}

for stop_idx in range(n_stops):
    model = xgb.XGBRegressor(**xgb_params)
    model.fit(X_train_scaled, y_train[:, stop_idx])
    pred = model.predict(X_test_scaled)
    y_pred_xgb_all.append(pred)

y_pred_xgb_all = np.array(y_pred_xgb_all).T

result_xgb = utils.evaluate_model(
    y_test, y_pred_xgb_all,
    model_name="XGBoost (Per Stop)",
    config={"params": xgb_params, "n_past_trips": n_past_trips, "scaler": "RobustScaler"}
)
evaluation_results.append(result_xgb)
print(result_xgb.summary())

In [ ]:
# Model Comparison Table and Save Results
utils.display_and_save_results(evaluation_results, const.EVALUATION_RESULTS_XGBOOST)